In [1]:
import equinox as eqx
import jax
import jax.numpy as jnp
from context_flux_no.models.multiphysics import (
    AbstractMultiphysicsOperator,
)
from context_flux_no.models.multiphysics.hyperfluxfno import (
    HyperNeuralOperator,
)
from jaxtyping import Array, Float, PRNGKeyArray


jax.config.update("jax_default_device", jax.devices("gpu")[4])

E0710 01:22:20.088037 3343278 platform_util.cc:270] Failed to create stream executor for device CUDA:0: : CUDA_ERROR_OUT_OF_MEMORY: out of memory


In [2]:
hyperfluxno = HyperNeuralOperator(
    num_spatial_dims=1,
    in_channels=1,
    in_timesteps=20,
    embedding_dim=128,
    encoder_type="TRecViT",
    encoder_kwargs=dict(
        grid_size=(100,),
        patch_size=(4,),
        depth=2,
        temporal_block_width=128,
        num_heads=8,
        mlp_hidden_dim=64,
    ),
    target_network_type="FluxNO",
    target_network_kwargs=dict(
        stencil_widths=(10, 10), lift_dim=128, hidden_dim=128, depth=4
    ),
    width_hyper=128,
    blocks_hyper=4,
    key=jax.random.key(0),
)
print(hyperfluxno.num_parameters() / 1e6)

/home/jhko725/projects/CONTEXT_FLUX_NO/src/context_flux_no/models/multiphysics/hyperfluxfno/utils.py:53: UserWarning: TRecViTEncoder supports variable in_timesteps. The given 
                    in_timesteps value will be ignored.
  warnings.warn(
/home/jhko725/projects/CONTEXT_FLUX_NO/src/context_flux_no/nn/structured_linear.py:40: UserWarning: out_features is not divisible by num_blocks. Output vector 
            will be truncated to the requested size.
  warnings.warn("""out_features is not divisible by num_blocks. Output vector


2.67802


In [8]:
hyperfluxno = HyperNeuralOperator(
    num_spatial_dims=1,
    in_channels=1,
    in_timesteps=20,
    embedding_dim=128,
    encoder_type="TRecViT",
    encoder_kwargs=dict(
        grid_size=(100,),
        patch_size=(4,),
        depth=2,
        temporal_block_width=128,
        num_heads=8,
        mlp_hidden_dim=64,
    ),
    target_network_type="FNO",
    target_network_kwargs=dict(
        frequency_modes=8, lift_dim=48, depth=4, width_lift=48, width_project=48
    ),
    width_hyper=128,
    blocks_hyper=4,
    key=jax.random.key(0),
)
print(hyperfluxno.num_parameters() / 1e6)

3.217636


In [10]:
hyperfluxno = HyperNeuralOperator(
    num_spatial_dims=1,
    in_channels=1,
    in_timesteps=20,
    embedding_dim=128,
    encoder_type="TRecViT",
    encoder_kwargs=dict(
        grid_size=(100,),
        patch_size=(4,),
        depth=2,
        temporal_block_width=128,
        num_heads=8,
        mlp_hidden_dim=64,
    ),
    target_network_type="UNet",
    target_network_kwargs=dict(hidden_channels_base=8, groups_norm=4, stack_grid=True),
    width_hyper=128,
    blocks_hyper=4,
    key=jax.random.key(0),
)
print(hyperfluxno.num_parameters() / 1e6)

4.01122


In [11]:
@eqx.filter_jit
def loss_fn(
    model: AbstractMultiphysicsOperator,
    u: Float[Array, "batch time dim ..."],
    args,
    key: PRNGKeyArray,
) -> tuple[Float[Array, ""], dict]:
    u0, u1 = u[:, :-1], u[:, -1]
    keys = jax.random.split(key, u0.shape[0])
    u1_pred: Float[Array, "batch dim ..."] = eqx.filter_vmap(
        lambda u_, key_: model(u_, args, key=key_)
    )(u0, keys)[0]
    return jnp.mean((u1 - u1_pred) ** 2), dict()


test_data = jax.random.normal(jax.random.key(0), (512, 21, 1, 100))

In [12]:
hyperfluxno(test_data[0, :20], (0.1, 0.01))

(20, 2, 100)


E0710 01:24:07.247439 3344893 xtile_compiler.cc:399] Fusion: gemm_fusion_dot = f32[128,500]{1,0} fusion(a.1, bitcast.14), kind=kCustom, calls=gemm_fusion_dot_computation.clone, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"fusion_backend_config":{"kind":"__triton_nested_gemm_fusion","block_level_fusion_config":{"num_warps":"8","output_tiles":[{"sizes":["128","256"]}],"num_ctas":1,"num_stages":4,"is_tma_allowed":false,"is_warp_specialization_allowed":false}},"force_earliest_schedule":false,"reification_cost":[],"device_type":"DEVICE_TYPE_INVALID"}
E0710 01:24:07.247536 3344893 xtile_compiler.cc:401] Computation: gemm_fusion_dot_computation.clone {
  parameter_0 = f32[128,128]{1,0} parameter(0)
  parameter_1 = f32[128,500]{0,1} parameter(1)
  ROOT dot.1 = f32[128,500]{1,0} dot(parameter_0, parameter_1), lhs_contracting_dims={1}, rhs_contracting_dims={0}, backend_config={"sizes":["32"]}
}
E0710 01:24:07.252341 3344857 xtile_compiler.cc:399] Fusion: gemm_fusion_do

(Array([[ 0.3117354 ,  1.5995171 ,  1.161767  ,  0.30145848,  0.71029204,
          2.2914884 , -0.5496999 , -0.5576248 ,  1.205263  ,  0.31447637,
          0.10655051,  0.60784316,  0.9410156 , -1.0602068 ,  0.95167434,
          1.2393031 ,  0.4524158 ,  0.5825163 , -0.01213515, -0.556567  ,
          0.3383975 ,  1.8390743 , -0.07142672, -0.83162093, -0.08063704,
         -0.90451646,  0.21428166, -0.48618174, -0.7169968 ,  0.00637545,
          1.0367059 , -0.71844435, -0.30706817, -1.0998288 ,  0.6573541 ,
         -0.81378806,  0.0200156 ,  2.0252435 ,  0.0799599 ,  1.2030272 ,
          1.1551961 ,  0.06607585,  0.07930022, -0.77886766, -0.99046254,
         -0.05651702,  0.25248617, -1.255871  , -0.2361033 , -0.7890959 ,
         -0.01263246,  0.94700885,  1.908508  , -0.8725474 ,  1.7357244 ,
         -0.62778145, -1.7014449 , -0.9022924 , -0.94335294, -1.3628522 ,
         -0.97977024,  0.7835078 ,  1.455822  , -1.0197797 , -1.0203825 ,
          0.6053051 ,  0.11628982, -0.

In [13]:
loss_fn(hyperfluxno, test_data, (0.1, 0.01), jax.random.key(0))

(20, 2, 100)


E0710 01:24:29.279987 3344839 xtile_compiler.cc:399] Fusion: gemm_fusion_dot.28 = f32[512,128]{1,0} fusion(mul.323, dynamic_nodonate__first___73_.1), kind=kCustom, calls=gemm_fusion_dot.28_computation.clone, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"fusion_backend_config":{"kind":"__triton_nested_gemm_fusion","block_level_fusion_config":{"num_warps":"8","output_tiles":[{"sizes":["128","256"]}],"num_ctas":1,"num_stages":4,"is_tma_allowed":false,"is_warp_specialization_allowed":false}},"force_earliest_schedule":false,"reification_cost":[],"device_type":"DEVICE_TYPE_INVALID"}
E0710 01:24:29.280087 3344839 xtile_compiler.cc:401] Computation: gemm_fusion_dot.28_computation.clone {
  parameter_0.27 = f32[512,128]{1,0} parameter(0)
  parameter_1.27 = f32[128,128]{1,0} parameter(1)
  ROOT dot.67 = f32[512,128]{1,0} dot(parameter_0.27, parameter_1.27), lhs_contracting_dims={1}, rhs_contracting_dims={1}, backend_config={"sizes":["32"]}
}
E0710 01:24:29.287892 334486

(Array(2.175629, dtype=float32), {})